# TinySLM Culinário v05 — reforçar C2 a partir da época 3

**Objetivo:** continuar a partir dos pesos da **época 3**, reforçando a extração C2 com os **2.272 exemplos incrementais** e uma pequena seleção de exemplos antigos (C1 e C3–C7). **Não recomeça o modelo do zero.**

### Preparar no Kaggle — três inputs

1. **Modelo anterior:** em **Add Input**, selecionar o output da execução v04 (com pasta `epoch_3`) **ou** um dataset com o ficheiro `TinySLM_Culinary_v04_continuacao_epoch_2_3.zip`. Não usar `epoca1`.
2. **Dataset v04 original:** manter o dataset `TinySLM Culinary Conversations v04` com `train.jsonl`, `validation.jsonl`, `dataset_report.json`.
3. **Novo dataset C2:** criar um dataset Kaggle privado com `train_c2_v05_completo.jsonl`, extraído do ZIP de entrega. Adicioná-lo em **Add Input**.

Nas definições do notebook ativar **GPU T4** (ou equivalente). Para executar sem manter a página aberta: **Save Version → Save & Run All**.

**Limites:** é um treino experimental, os exemplos foram verificados estruturalmente mas nem todos manualmente; a avaliação v04 não substitui um benchmark independente C2. O checkpoint contém pesos, não estado de AdamW: reiniciamos o otimizador, por isso não é uma retoma bit-a-bit.

## 1. Bibliotecas, dispositivo e parâmetros

- Mantemos o tokenizer do modelo anterior.
- Usamos uma taxa de aprendizagem mais baixa e **2 épocas curtas**.
- A mistura inclui 600 conversas de cada competência antiga C1/C3–C7 (até 3.600) e os novos exemplos C2 que cabem no contexto.
- Os inputs nunca são alterados; tudo o que é produzido fica em `/kaggle/working`.

In [ ]:
import os, sys, json, math, time, random, hashlib, collections, shutil, zipfile
from pathlib import Path
import numpy as np
import torch
import transformers
from transformers import AutoTokenizer, LlamaForCausalLM
from torch.optim import AdamW
from torch.utils.data import DataLoader

print('Python:',sys.version.split()[0], '| Torch:',torch.__version__, '| Transformers:',transformers.__version__)
assert torch.cuda.is_available(), 'Ative uma GPU no painel do Kaggle antes de continuar.'
print('GPU:',torch.cuda.get_device_name(0))

SEED=42
MAX_LEN=512
EPOCHS=2
LR=1e-5
MICRO_BATCH=4
GRAD_ACCUM=4
REHEARSAL_PER_CAP=600  # C1, C3, C4, C5, C6, C7
MAX_NEW_TOKENS=96

OUT=Path('/kaggle/working/tinyslm_c2_v05')
OUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


## 2. Localizar automaticamente os três inputs

Aceita a pasta `epoch_3` diretamente ou o ZIP da execução anterior. **O ZIP tem pastas**, por isso não pode ser aberto como se fosse `model.safetensors`.

Se aparecer `AssertionError`, verificar no painel **Input** se os três datasets foram adicionados. A célula mostra o que encontrou.

In [ ]:
INPUT=Path('/kaggle/input')
assert INPUT.exists(), 'O notebook tem de correr no Kaggle.'

def exactly_one(paths,label):
    paths=list(paths)
    assert len(paths)==1, f'{label}: esperado um resultado, encontrados: {paths[:10]}'
    return paths[0]

# 1. Preferir pesos da época 3 disponíveis numa pasta de outputs.
direct=[p.parent for p in INPUT.rglob('model.safetensors') if p.parent.name=='epoch_3']
if direct:
    BASE=exactly_one(direct,'Pastas epoch_3')
else:
    previous=[p for p in INPUT.rglob('*.zip') if 'continuacao_epoch_2_3' in p.name.lower()]
    archive_path=exactly_one(previous,'ZIP do treino v04 com epoch_3')
    BASE=OUT/'input_epoch_3'
    BASE.mkdir(exist_ok=True)
    required={'model.safetensors','config.json','tokenizer.json','tokenizer_config.json'}
    with zipfile.ZipFile(archive_path) as z:
        available={Path(n).name:n for n in z.namelist() if n.startswith('epoch_3/') and not n.endswith('/')}
        assert required.issubset(available), f'epoch_3 incompleta: {required-set(available)}'
        assert z.testzip() is None, 'ZIP de pesos danificado'
        for name,member in available.items():
            if name in required|{'generation_config.json'}:
                with z.open(member) as src, (BASE/name).open('wb') as dst:shutil.copyfileobj(src,dst)

reports=list(INPUT.rglob('dataset_report.json'))
DATA_DIR=exactly_one([p.parent for p in reports if (p.parent/'train.jsonl').is_file() and (p.parent/'validation.jsonl').is_file()],
                     'Dataset v04 original')

new_files=list(INPUT.rglob('train_c2_v05_completo.jsonl'))
if new_files:
    C2_FILE=exactly_one(new_files,'Dataset C2 v05')
else:
    candidates=[p for p in INPUT.rglob('*.zip') if 'c2_v05_completo' in p.name.lower()]
    source=exactly_one(candidates,'ZIP do dataset C2 v05')
    C2_FILE=OUT/'train_c2_v05_completo.jsonl'
    with zipfile.ZipFile(source) as z:
        member=exactly_one([n for n in z.namelist() if Path(n).name=='train_c2_v05_completo.jsonl'], 'Ficheiro C2 dentro do ZIP')
        with z.open(member) as src,C2_FILE.open('wb') as dst:shutil.copyfileobj(src,dst)

print('Checkpoint da época 3:',BASE)
print('Dataset original:',DATA_DIR)
print('Novos exemplos C2:',C2_FILE)
assert all((BASE/n).is_file() for n in ('model.safetensors','config.json','tokenizer.json','tokenizer_config.json'))


## 3. Carregar o modelo e construir a mistura de treino

Para limitar memória, percorremos o dataset antigo linha a linha e guardamos apenas uma amostra aleatória por competência. Os **2.272 exemplos C2 novos** entram como candidatos sem substituir os anteriores.

Se uma conversa ultrapassar os 512 tokens do contexto, é excluída **antes** do treino; o relatório mostra as contagens reais.

In [ ]:
tok=AutoTokenizer.from_pretrained(BASE,local_files_only=True)
assert len(tok)==8005, 'Tokenizer não corresponde ao checkpoint v04 de 8005 tokens.'
assert tok.convert_tokens_to_ids('<|user|>')==8000
assert tok.convert_tokens_to_ids('<|assistant|>')==8001
if tok.pad_token_id is None:tok.pad_token=tok.eos_token

model=LlamaForCausalLM.from_pretrained(BASE,local_files_only=True,torch_dtype=torch.float32)
assert model.get_input_embeddings().weight.shape[0]==len(tok)
assert MAX_LEN<=model.config.max_position_embeddings
model.config.pad_token_id=tok.pad_token_id
device=torch.device('cuda')
model.to(device)
AMP_DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler=torch.amp.GradScaler('cuda',enabled=AMP_DTYPE==torch.float16)
print('Modelo carregado, parâmetros:',sum(p.numel() for p in model.parameters()))

def read_jsonl(path):
    with Path(path).open() as f:
        for line in f:
            if line.strip():yield json.loads(line)

old_caps=('C1','C3','C4','C5','C6','C7')
rng=random.Random(SEED)
reservoir={cap:[] for cap in old_caps}
seen=collections.Counter()
for row in read_jsonl(DATA_DIR/'train.jsonl'):
    cap=row.get('capability')
    if cap not in reservoir:continue
    seen[cap]+=1
    items=reservoir[cap]
    if len(items)<REHEARSAL_PER_CAP:items.append(row)
    else:
        j=rng.randrange(seen[cap])
        if j<REHEARSAL_PER_CAP:items[j]=row
old_rows=[r for cap in old_caps for r in reservoir[cap]]
c2_rows=list(read_jsonl(C2_FILE))
assert len(c2_rows)==2272, f'Eram esperados 2272 exemplos C2, encontrados {len(c2_rows)}'
assert all(r.get('capability')=='C2' and r.get('split')=='train' for r in c2_rows)
train_rows=old_rows+c2_rows
val_rows=list(read_jsonl(DATA_DIR/'validation.jsonl'))
assert set(r['recipe_id'] for r in train_rows).isdisjoint(r['recipe_id'] for r in val_rows), 'Há receitas comuns ao treino e à validação!'
report=json.loads((DATA_DIR/'dataset_report.json').read_text())
assert hashlib.sha256((DATA_DIR/'validation.jsonl').read_bytes()).hexdigest()==report['validation_sha256'], 'A validação v04 foi alterada.'
print('Amostra antiga:',len(old_rows),'| Novos C2:',len(c2_rows),'| Validação:',len(val_rows))


## 4. Preparar tokens apenas para a resposta do assistente

O modelo recebe **receita + pergunta**, mas a perda (*loss*) incide apenas sobre a resposta correta. Evitamos treinar o modelo para repetir o enunciado ou a receita.

In [ ]:
SYSTEM="Answer in English using the supplied recipe. Follow the user's request. If the recipe does not establish an answer, state the missing or conflicting information."
def enc(s):return tok.encode(s,add_special_tokens=False)
def prompt_ids(row,assistant_index):
    ids=[tok.bos_token_id]+enc('<|system|>\n'+SYSTEM+'\n')
    context='\n\n'.join(c['text'] for c in row['context']['chunks'])
    for i,msg in enumerate(row['conversation'][:assistant_index]):
        content=msg['content']
        if i==0:content='Recipe context:\n'+context+'\n\nQuestion:\n'+content
        ids+=enc('<|'+msg['role']+'|>\n'+content)
        if msg['role']=='assistant':ids.append(tok.eos_token_id)
        ids+=enc('\n')
    return ids+enc('<|assistant|>\n')

def prepare(rows,label):
    samples=[];counts=collections.Counter();total=collections.Counter()
    for row in rows:
        for i,msg in enumerate(row['conversation']):
            if msg['role']!='assistant':continue
            cap=row['capability'];total[cap]+=1
            prompt=prompt_ids(row,i)
            target=enc(msg['content'])+[tok.eos_token_id]
            if len(prompt)+len(target)>MAX_LEN:continue
            counts[cap]+=1
            samples.append({'input_ids':prompt+target,'labels':[-100]*len(prompt)+target,
                            'prompt':prompt,'row':row,'assistant_index':i})
    assert samples, 'Nenhuma conversa cabe no contexto.'
    print(label,'admitidas:',dict(counts),'| total:',dict(total))
    return samples

train_samples=prepare(train_rows,'Treino')
val_samples=prepare(val_rows,'Validação')
assert sum(s['row']['capability']=='C2' for s in train_samples)>100, 'Poucos exemplos C2 cabem nos 512 tokens.'

def collate(group):
    size=max(len(x['input_ids']) for x in group)
    ids=torch.full((len(group),size),tok.pad_token_id,dtype=torch.long)
    labels=torch.full_like(ids,-100)
    mask=torch.zeros_like(ids)
    for i,s in enumerate(group):
        n=len(s['input_ids'])
        ids[i,:n]=torch.tensor(s['input_ids'])
        labels[i,:n]=torch.tensor(s['labels'])
        mask[i,:n]=1
    return {'input_ids':ids,'labels':labels,'attention_mask':mask}

train_loader=DataLoader(train_samples,batch_size=MICRO_BATCH,shuffle=True,
                        generator=torch.Generator().manual_seed(SEED),collate_fn=collate,num_workers=0)
val_loader=DataLoader(val_samples,batch_size=MICRO_BATCH,collate_fn=collate,num_workers=0)
print('Batches por época:',len(train_loader))


## 5. Avaliar o checkpoint **antes** do novo treino

Esta medição fixa uma comparação antes/depois sobre a mesma validação v04. Guardamos também algumas respostas de geração com contexto completo para analisar manualmente. O desempenho em C2 só fica comprovado depois de um teste independente.

In [ ]:
@torch.no_grad()
def evaluate():
    model.eval();total_nll=0.;total_tokens=0
    for batch in val_loader:
        batch={k:v.to(device) for k,v in batch.items()}
        n=int((batch['labels'][:,1:]!=-100).sum())
        with torch.autocast(device_type='cuda',dtype=AMP_DTYPE):
            loss=model(**batch,use_cache=False).loss
        assert torch.isfinite(loss), 'Loss de validação não finita.'
        total_nll+=float(loss)*n;total_tokens+=n
    result=total_nll/total_tokens
    return {'validation_loss':result,'assistant_perplexity':math.exp(min(result,20)),'target_tokens':total_tokens}

# Exemplos fixos, incluindo C2; excluímos prompts sem espaço para a resposta.
eval_cases=[]
for cap in sorted(set(s['row']['capability'] for s in val_samples)):
    choices=[s for s in val_samples if s['row']['capability']==cap and len(s['prompt'])+MAX_NEW_TOKENS<=MAX_LEN]
    choices.sort(key=lambda s:s['row']['id'])
    chosen=set()
    for s in choices:
        if s['row']['recipe_id'] in chosen:continue
        eval_cases.append(s);chosen.add(s['row']['recipe_id'])
        if len(chosen)>=2:break
    print(cap,'perguntas de comparação:',len(chosen))

@torch.no_grad()
def generation_suite(stage):
    model.eval();out=[]
    stops=list(set([tok.eos_token_id]+[tok.convert_tokens_to_ids(t) for t in ('<|user|>','<|assistant|>','<|system|>','<|tool|>','<|tool_result|>')]))
    for s in eval_cases:
        prompt=torch.tensor([s['prompt']],device=device)
        with torch.autocast(device_type='cuda',dtype=AMP_DTYPE):
            generated=model.generate(input_ids=prompt,attention_mask=torch.ones_like(prompt),
                                     max_new_tokens=MAX_NEW_TOKENS,do_sample=False,
                                     eos_token_id=stops,pad_token_id=tok.pad_token_id)
        row=s['row'];i=s['assistant_index']
        predicted=tok.decode(generated[0,len(s['prompt']):],skip_special_tokens=True)
        out.append({'stage':stage,'capability':row['capability'],'id':row['id'],
                    'recipe':row['context'],'question':row['conversation'][i-1]['content'],
                    'reference':row['conversation'][i]['content'],'prediction':predicted})
        print(stage,row['capability'],row['conversation'][i-1]['content'],'\n=>',predicted,'\n')
    return out

history=[{'stage':'antes_c2_v05','epoch':0,**evaluate()}]
generations=generation_suite('antes_c2_v05')
def save_reports():
    (OUT/'metrics.json').write_text(json.dumps(history,indent=2))
    (OUT/'generations_comparacao.json').write_text(json.dumps(generations,ensure_ascii=False,indent=2))
    (OUT/'config.json').write_text(json.dumps({'epochs':EPOCHS,'lr':LR,'rehearsal_per_cap':REHEARSAL_PER_CAP,
        'original_data':str(DATA_DIR),'c2_data':str(C2_FILE),'checkpoint_epoch3':str(BASE),
        'validation_sha256':report['validation_sha256']},indent=2))
save_reports()
print('Antes:',history[0])


## 6. Continuar o treino e guardar os checkpoints

- O otimizador é novo (não temos o seu estado da época 3).
- Cada época tem um checkpoint `epoch_c2_1`, `epoch_c2_2`.
- O notebook guarda resultados **após cada época**, para ser possível usar uma versão parcial se a execução for interrompida.
- O melhor checkpoint é escolhido pela perda de validação, não pelo número de épocas.

In [ ]:
optimizer=AdamW(model.parameters(),lr=LR,betas=(.9,.95),weight_decay=.1)
updates=math.ceil(len(train_loader)/GRAD_ACCUM)*EPOCHS
warmup=max(1,round(.05*updates))
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda step:min(1.0,(step+1)/warmup))
best_loss=history[0]['validation_loss']
(OUT/'best_checkpoint.json').write_text(json.dumps({'stage':'antes_c2_v05','validation_loss':best_loss},indent=2))

for epoch in range(1,EPOCHS+1):
    model.train();start=time.time();total_nll=0.;total_tokens=0;group_tokens=0
    optimizer.zero_grad(set_to_none=True)
    for batch_no,batch in enumerate(train_loader,1):
        batch={k:v.to(device) for k,v in batch.items()}
        n=int((batch['labels'][:,1:]!=-100).sum())
        with torch.autocast(device_type='cuda',dtype=AMP_DTYPE):
            loss=model(**batch,use_cache=False).loss
        if not torch.isfinite(loss):raise RuntimeError('Loss não finita; parar treino.')
        scale_tokens=MICRO_BATCH*MAX_LEN*GRAD_ACCUM
        scaler.scale(loss*n/scale_tokens).backward()
        total_nll+=float(loss.detach())*n;total_tokens+=n;group_tokens+=n
        if batch_no%GRAD_ACCUM==0 or batch_no==len(train_loader):
            scaler.unscale_(optimizer)
            for parameter in model.parameters():
                if parameter.grad is not None:parameter.grad.mul_(scale_tokens/group_tokens)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            old_scale=scaler.get_scale()
            scaler.step(optimizer);scaler.update()
            if scaler.get_scale()>=old_scale:scheduler.step()
            optimizer.zero_grad(set_to_none=True);group_tokens=0
        if batch_no%100==0 or batch_no==len(train_loader):
            mins=(time.time()-start)/60
            eta=mins*(len(train_loader)-batch_no)/batch_no
            print(f'C2 época {epoch}: {batch_no}/{len(train_loader)} | loss {total_nll/total_tokens:.4f} | ETA {eta:.1f} min',flush=True)

    metrics={'stage':f'epoch_c2_{epoch}','epoch':epoch,'train_loss':total_nll/total_tokens,
             'minutes':(time.time()-start)/60,**evaluate()}
    history.append(metrics)
    checkpoint=OUT/f'epoch_c2_{epoch}'
    model.save_pretrained(checkpoint,safe_serialization=True)
    tok.save_pretrained(checkpoint)
    if metrics['validation_loss']<best_loss:
        best_loss=metrics['validation_loss']
        (OUT/'best_checkpoint.json').write_text(json.dumps({'checkpoint':checkpoint.name,**metrics},indent=2))
    generations.extend(generation_suite(f'epoch_c2_{epoch}'))
    save_reports()
    print(metrics,flush=True)


## 7. Exportar os resultados

No final, os ficheiros ficam em **Output**, dentro de `tinyslm_c2_v05`. É criado também um único ZIP. O modelo **não** é publicado automaticamente e os datasets originais não são substituídos.

In [ ]:
import zipfile
archive=Path('/kaggle/working/TinySLM_C2_v05_treino_resultados.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for path in OUT.rglob('*'):
        if path.is_file():z.write(path,path.relative_to(OUT))
print('Exportado:',archive,'| MB:',round(archive.stat().st_size/1e6,1))
print('Ver resultados em Save Version → Output. Não confundir com os inputs.')


In [ ]:
# ============================================================
# Carregar checkpoint base — época 3
# ============================================================

from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Procurar checkpoints no Kaggle
candidates = []

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    if root.exists():
        for config_file in root.rglob("config.json"):
            folder = config_file.parent

            if (folder / "model.safetensors").exists():
                candidates.append(folder)

print("Checkpoints encontrados:")

for i, folder in enumerate(candidates):
    print(i, folder)

# Selecionar especificamente epoch_c2_2
matches = [
    folder for folder in candidates
    if folder.name == "epoch_3"
]

if len(matches) != 1:
    raise RuntimeError(
        f"Esperava encontrar um checkpoint epoch_3, "
        f"mas encontrei {len(matches)}. Verifica os caminhos acima."
    )

MODEL_PATH = matches[0]

print("\nCheckpoint selecionado:", MODEL_PATH)

# Carregar tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=False
)

# Carregar modelo
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=False,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = model.to(device)
model.eval()

print("\nModelo carregado com sucesso!")
print("Dispositivo:", device)
print("Parâmetros:", sum(p.numel() for p in model.parameters()))

In [ ]:
# ============================================================
# TinySLM C2 — Teste de identificação de intenções
# Sem contexto da receita e sem treino adicional
# ============================================================

import torch
import time

# Verificar se o modelo está carregado
assert "model" in globals(), "O modelo não está carregado."
assert "tokenizer" in globals(), "O tokenizer não está carregado."

model.eval()

device = next(model.parameters()).device

print("Modelo disponível:", type(model).__name__)
print("Dispositivo:", device)

# Perguntas para testar diferentes intenções
TEST_QUESTIONS = [
    # INGREDIENTES
    ("What ingredients do I need?", "ingredients"),
    ("Which spices should I use?", "ingredients"),
    ("Does this recipe contain garlic?", "ingredients"),

    # QUANTIDADES
    ("How much paprika should I use?", "quantity"),
    ("How many potatoes do I need?", "quantity"),
    ("What amount of butter is required?", "quantity"),

    # PASSOS
    ("What should I do after boiling the potatoes?", "steps"),
    ("When should I add the cauliflower?", "steps"),
    ("How do I prepare the onions?", "steps"),

    # RESTRIÇÕES
    ("Can I make this recipe without butter?", "constraints"),
    ("Does this recipe meet a dairy-free requirement?", "constraints"),
    ("I cannot eat eggs. Can I follow this recipe unchanged?", "constraints"),

    # SITUAÇÕES MAIS COMPLEXAS
    ("I have cumin and turmeric. What else do I need?", "ingredients"),
    ("What can I use instead of cream cheese?", "constraints"),

    # FORA DO DOMÍNIO
    ("Who won the football match yesterday?", "unknown"),
]

# Instrução curta para identificação da intenção
INTENT_PROMPT = """Classify the user's question.

Return only one of these labels:
ingredients
quantity
steps
constraints
unknown

Question: {question}
Intent:"""


@torch.no_grad()
def test_intent(question):

    prompt = INTENT_PROMPT.format(question=question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        return_token_type_ids=False
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=15,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # Descodificar apenas os tokens gerados
    generated = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return response


# Executar testes
results = []

print("\n" + "=" * 70)
print("TinySLM C2 — AVALIAÇÃO DE INTENÇÕES")
print("=" * 70)

start = time.time()

for number, (question, expected) in enumerate(TEST_QUESTIONS, 1):

    prediction = test_intent(question)

    # Comparação estrita: só aceitamos o rótulo exato
    correct = prediction.lower() == expected.lower()

    results.append({
        "question": question,
        "expected": expected,
        "prediction": prediction,
        "correct": correct
    })

    print(f"\nTESTE {number}")
    print("Pergunta :", question)
    print("Esperado :", expected)
    print("Modelo   :", repr(prediction))
    print("Resultado:", "OK" if correct else "ERRO")

# Resumo
total = len(results)
correct_count = sum(r["correct"] for r in results)

print("\n" + "=" * 70)
print("RESUMO")
print("=" * 70)

print("Perguntas:", total)
print("Corretas :", correct_count)
print("Precisão :", f"{correct_count / total * 100:.1f}%")
print("Tempo    :", f"{time.time() - start:.1f} segundos")

## 8. Dataset de intenções v02 — carregar sem repetir o treino C2

**Continuação experimental do mesmo notebook.** As secções 1–7 continuam intactas, com o treino C2 anterior; a secção 7 exporta os resultados originais. As células 8–10 são independentes e não devem obrigar a repetir as épocas anteriores.

O novo dataset está versionado em `smartlearningci/tinyLLM/data/intent_candidates/`. Contém 24.000 perguntas de treino, 3.000 de validação, 3.000 de teste e casos para revisão.

**ATENÇÃO:** os rótulos de intenção/operação são *candidatos heurísticos*, ainda não confirmados manualmente. Não vamos tratar a validação automática como prova de generalização. As classes e operações ainda não incluem `unknown` e extração de argumentos com fiabilidade. Primeiro vamos inspecionar amostras e fazer uma auditoria.

No Kaggle, podes: **(A)** adicionar o ZIP como *Input*, ou **(B)** ativar Internet e deixar a célula descarregá-lo diretamente do GitHub. A célula abaixo é independente de `DATA_DIR`, `C2_FILE` e de `train_loader`.

In [ ]:
# 8.1 — Localizar o ZIP com os candidatos (sem tocar nos checkpoints)
from pathlib import Path
import urllib.request, zipfile, json, hashlib, shutil

INTENT_OUTPUT = Path('/kaggle/working/tinyslm_intents_v02')
INTENT_OUTPUT.mkdir(parents=True, exist_ok=True)
INTENT_ARCHIVE_NAME = 'TinySLM_C2_intents_candidates_v02.zip'
INTENT_SHA256 = '82ad978151539f37e356a5591c97bb51058bbbbf63ae7d2b012b989e347aa6dd'
INTENT_URL = ('https://raw.githubusercontent.com/smartlearningci/tinyLLM/main/'
              'data/intent_candidates/' + INTENT_ARCHIVE_NAME)

candidates = sorted(set(Path('/kaggle/input').rglob(INTENT_ARCHIVE_NAME)))
if len(candidates) > 1:
    raise RuntimeError(f'Vários ZIP de intenções disponíveis: {candidates}')
if candidates:
    archive = candidates[0]
else:
    archive = INTENT_OUTPUT / INTENT_ARCHIVE_NAME
    if not archive.exists():
        print('O dataset não está em Add Input. A descarregar do GitHub público...')
        try:
            urllib.request.urlretrieve(INTENT_URL, archive)
        except Exception as exc:
            raise RuntimeError('Adiciona o ZIP das intenções como Input no Kaggle ou ativa Internet.') from exc

digest = hashlib.sha256(archive.read_bytes()).hexdigest()
assert digest == INTENT_SHA256, f'Checksum não corresponde: {digest}'
print('Dataset encontrado e validado:', archive)

INTENT_DIR = INTENT_OUTPUT / 'dataset'
INTENT_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    required = {'intent_dataset_v02/train.jsonl', 'intent_dataset_v02/validation.jsonl',
                'intent_dataset_v02/test.jsonl', 'intent_dataset_v02/review.jsonl',
                'intent_dataset_v02/build_report.json'}
    assert required.issubset(set(z.namelist())), 'ZIP incompleto.'
    for filename in sorted(required):
        target = INTENT_DIR / Path(filename).name
        with z.open(filename) as reader, target.open('wb') as writer:
            shutil.copyfileobj(reader, writer)

report_intents = json.loads((INTENT_DIR / 'build_report.json').read_text())
print('Estado:', report_intents['status'])
print('Contagens:', report_intents['counts'])

## 9. Auditoria de qualidade antes do refinamento

Os candidatos desta versão são **fracos**, não são exemplos confirmados. Esta secção mostra uma amostra estratificada por intenção/operação, cria um ficheiro CSV para revisão e verifica o isolamento por receita. Não apresenta a precisão dos rótulos como se fosse uma medição humana.

As perguntas ambíguas e não reconhecidas permanecem no ficheiro `review.jsonl`. É particularmente importante rever `substitute`, `missing` e questões com múltiplas intenções.

In [ ]:
# 9.1 — Distribuição, amostras e revisão exportável
import collections, random, csv

def read_rows(path):
    with Path(path).open(encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)

intent_rows = {split: list(read_rows(INTENT_DIR / f'{split}.jsonl'))
               for split in ('train', 'validation', 'test')}
by_group = collections.Counter((r['target']['intent'], r['target']['operation'])
                               for r in intent_rows['train'])
print('Categorias/ operações de treino:')
for (intent, operation), count in sorted(by_group.items()):
    print(f'  {intent:12s} {operation:12s} {count:5d}')

recipe_sets = {split: {r['recipe_id'] for r in rows} for split, rows in intent_rows.items()}
for a, b in [('train','validation'),('train','test'),('validation','test')]:
    assert not (recipe_sets[a] & recipe_sets[b]), f'Receitas repetidas entre {a} e {b}'
print('Isolamento por receita: OK')

# Para cada combinação, guardamos até 20 perguntas na revisão inicial.
rng = random.Random(2026)
groups = collections.defaultdict(list)
for r in intent_rows['train']:
    groups[(r['target']['intent'], r['target']['operation'])].append(r)
sample = []
for group, rows in sorted(groups.items()):
    sample.extend(rng.sample(rows, min(20, len(rows))))

review_csv = INTENT_OUTPUT / 'amostra_revisao_180.csv'
with review_csv.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.writer(handle)
    writer.writerow(['question','intent_candidate','operation_candidate','correct_yes_no','corrected_intent','corrected_operation','notes'])
    for row in sample:
        writer.writerow([row['question'], row['target']['intent'], row['target']['operation'], '', '', '', ''])
print('Amostras para rever:', len(sample), '| CSV:', review_csv)
for row in sample[:12]:
    print(row['target'], '—', row['question'])

## 10. Preparar o refinamento do mesmo TinySLM — fase seguinte

O nosso objetivo continua a ser **um único TinySLM em duas passagens**: primeiro interpreta a pergunta sem receita, depois recebe os dados da ferramenta e responde.

Formato inicialmente proposto da primeira passagem:

```text
<|system|>
Interpret the user's cooking request. Return only JSON with intent and operation.
<|user|>
How much paprika should I use?
<|assistant|>
{"intent":"quantity","operation":"get"}
```

**O treino ainda não é executado aqui**, porque os rótulos no ZIP são provisórios e a extração de argumentos ainda falta. A célula prepara e testa a codificação de exemplos para confirmar compatibilidade com o tokenizer e o checkpoint C2. Depois de validarmos as etiquetas, acrescentaremos o treino com mistura de exemplos culinários antigos, avaliação humana independente e gravação num novo checkpoint. O checkpoint original nunca será sobrescrito.

In [ ]:
# 10.1 — Preparar exemplos para fine-tuning sem executar backpropagation
INTENT_SYSTEM = 'Interpret the cooking question. Return only valid JSON with intent and operation.'

def get_loaded_tokenizer():
    if 'tok' in globals() and tok is not None:
        return tok
    if 'tokenizer' in globals() and tokenizer is not None:
        return tokenizer
    raise RuntimeError('Carrega primeiro o tokenizer do checkpoint epoch_3 (célula 15).')

def encode_intent_pair(row):
    tokenizer_ = get_loaded_tokenizer()
    prompt = ('<|system|>\n' + INTENT_SYSTEM + '\n'
              '<|user|>\n' + row['question'] + '\n'
              '<|assistant|>\n')
    target = json.dumps(row['target'], ensure_ascii=False, separators=(',',':'))
    prompt_ids = tokenizer_.encode(prompt, add_special_tokens=False)
    answer_ids = tokenizer_.encode(target, add_special_tokens=False)
    if tokenizer_.eos_token_id is not None:
        answer_ids.append(tokenizer_.eos_token_id)
    return {'input_ids': prompt_ids + answer_ids,
            'labels': [-100] * len(prompt_ids) + answer_ids,
            'prompt': prompt, 'target': target,
            'prompt_token_count': len(prompt_ids)}

example = encode_intent_pair(intent_rows['train'][0])
print('Exemplo de treino preparado:')
print(example['prompt'])
print(example['target'])
print('Tokens:', len(example['input_ids']))
assert len(example['input_ids']) <= 512, 'Este exemplo excede o contexto do TinySLM.'
assert all(v == -100 for v in example['labels'][:example['prompt_token_count']])
assert all(v >= 0 for v in example['labels'][example['prompt_token_count']:])

RUN_INTENT_FINETUNE = False  # PROTEÇÃO: não ativar antes da validação humana.
if RUN_INTENT_FINETUNE:
    raise RuntimeError('Refinamento bloqueado: é necessária uma versão validada dos rótulos.')
print('Treino de intenções: ainda não executado. Checkpoint C2 preservado.')